In [2]:
import networkx as nx
import matplotlib.pyplot as plt
from random import randint
from matplotlib.lines import Line2D

In [12]:
#Generate Grid
def GenGrid(l,w):
    Grid = []
    for i in range(l*w):
        Sub = []
        for j in range(w*l):
            if ((j == i - 1) or (j == i + 1)
                or (j == i + w) or (j == i - w)):
                Sub.append(1)
            else:
                Sub.append(0)
            if ((i+1)%w == 0 and j == i + 1) or (i%w == 0 and j == i - 1):
                Sub[j] = 0
        Grid.append(Sub)
    return Grid
            

In [13]:
def VisualiseRoute(adj_matrix, path, l, w, arrow_edges):
    # Create the graph from the adjacency matrix
    G = nx.Graph()
    for i, row in enumerate(adj_matrix):
        for j, val in enumerate(row):
            if val == 1:
                G.add_edge(i, j)
    pos = {}
    for node in G.nodes():
        row = node // w
        col = node % w
        pos[node] = (col, -row)

    path_edges = list(zip(path, path[1:]))

    if arrow_edges is None:
        arrow_edges = []

    node_colors = []
    for node in G.nodes():
        if node == 0:
            node_colors.append("limegreen")
        elif node in path:
            node_colors.append("orange")
        else:
            node_colors.append("skyblue")

    plt.figure(figsize=(8, 6))
    nx.draw(G, pos,
            with_labels=True,
            node_color=node_colors,
            node_size=500,
            font_weight="bold",
            edge_color="gray")

    nx.draw_networkx_edges(G, pos,
                           edgelist=path_edges,
                           edge_color="red",
                           width=3,
                          arrows=True,
                            arrowstyle="-|>",
                            arrowsize=20,
                            connectionstyle="arc3,rad=0.0")

    if arrow_edges:
        nx.draw_networkx_edges(
            G, pos,
            edgelist=arrow_edges,
            edge_color="blueviolet",  
            width=3,
            arrows=True,
            arrowstyle="-|>",
            arrowsize=20,
            connectionstyle="arc3,rad=0.0"
        )
    
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Destination Node',
               markerfacecolor='limegreen', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Route Node',
               markerfacecolor='orange', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Other Node',
               markerfacecolor='skyblue', markersize=10),
        Line2D([0], [0], color='red', lw=3, label='Route Edge'),
        Line2D([0], [0], color='blueviolet', lw=3, label='Potential Swaps')
    ]

    plt.legend(handles=legend_elements,
           loc='upper left',
           bbox_to_anchor=(1.02, 1),
           borderaxespad=0.)
    plt.axis("equal")
    plt.savefig("Graph.png")
    plt.show()

In [14]:
#Random Passenger Demand Grid
def RandomDemand(l,w, Range):
    Passenger = []
    for i in range(l):
        for j in range(w):
            Passenger.append(randint(0,Range))
    return Passenger

In [15]:
#Fixed Passenger Demand Grid
def FixedDemand(l,w, D):
    Passenger = [D] * l * w
    return Passenger 

In [16]:
def CheckNeighbours(adj_matrix,path,Node,Checked):
    
    Stop = []
    x1 = []
    
    for i in range(len(adj_matrix)):
        if adj_matrix[Node][i] == 1 and not i in Checked:
            Checked.append(i)
            x1.append(i)
            if i in path:
                Stop.append(i)
    
    return x1, Stop, Checked

In [17]:
#Passenger pathing
def NearestStop(adj_matrix,path,Node):

    #Check the node isn't a stop and load lists
    Walk = [Node]
    if Node in path:
        return Walk
    Checked = [Node]
    Stop = []

    #Find the nearest stop(s)
    Neighbours = [[Node]]
    n = 0
    while len(Stop) == 0:

        #Initialise List
        Neighbours.append([])
        #Check all the neighbours
        for i in Neighbours[n]:
            x,y, Checked = CheckNeighbours(adj_matrix,path,i,Checked)
            for j in x:
                Neighbours[n+1].append(j)
            
            for k in y:
                Stop.append(k)
        n += 1
        if n >= 100:
            break

    #Find the best stop
    best = Stop[0]
    for i in Stop:
        if path.index(i) > path.index(best):
            best = i

    #Find the path from start to best stop
    Walk.insert(-1,best)
    for i in range(2,len(Neighbours)+1):
        for j in Neighbours[-i]:
            if adj_matrix[Walk[-i]][j] == 1:
                Walk.insert(-i,j)
                break
    Walk.pop(-1)
    return Walk


In [18]:
#Bus Cost calculation
def BusCost(adj_matrix,path,k1):
    return k1*(len(path)-1)

In [19]:
#Passenger Cost calculation
def PassengerCost(adj_matrix,Demand,path, k2, k3,k4):
    total = 0
    n = 0
    for i in Demand:
        Walk = NearestStop(adj_matrix,path,n)
        Bus = len(path)-1 - path.index(Walk[-1])
        if Bus > 0:
            total += k4 * i
        total += i * k2 * (len(Walk)-1) + i * k3 * Bus
        n += 1
    return total

In [20]:
def TotalCost(adj_matrix,Demand,path, k1,k2,k3,k4):
    TotalCost = PassengerCost(adj_matrix,Demand,path, k2, k3, k4) + BusCost(adj_matrix,path,k1)
    return TotalCost

In [21]:
def FindNeighbours(adj_matrix, Node, route):
    Neighbours = []
    for i in range(len(adj_matrix)):
        if adj_matrix[Node][i] == 1 and not i in route:
            Neighbours.append(i)
    return Neighbours

In [22]:
def ExtendRoute2(adj_matrix, Destination, Checked, route, New):
    Tree = FindNeighbours(adj_matrix, route[-1], route)
    if len(Tree) > 0:
        for i in Tree:
            if not (str(route),i) in Checked:
                if Destination == i:
                    route.append(Destination)
                    New.append(route[:])
                    Checked[str(route[0:-1]),i] = 1
                    route.pop(-1)
                else:
                    route.append(i)
                    ExtendRoute2(adj_matrix, Destination, Checked, route, New)
    
    Checked[str(route[0:-1]),route[-1]] = 1
    route.pop(-1)
    
    return New
        

In [23]:
def ExhaustiveSearch(adj_matrix,Demand, k1,k2,k3,k4, Destination):
    Routes = []
    Checked = {}
    New = []
    for i in range(len(adj_matrix)):
        if i == Destination:
            Routes.append([i])
        else:
            route = [i]
            New = ExtendRoute2(adj_matrix, Destination, Checked, route, New)
            for j in New:
                if not j in Routes:
                    Routes.append(j)
    return Routes


In [24]:
def ExhaustiveOptimal(adj_matrix, Demand, k1,k2,k3,k4, Destination):
    Routes = ExhaustiveSearch(adj_matrix, Demand, k1,k2,k3,k4, Destination)
    best = [[Destination],TotalCost(adj_matrix,Demand,[Destination], k1,k2,k3,k4)]
    extras = []
    for i in Routes:
        RouteCost = TotalCost(adj_matrix,Demand,i, k1,k2,k3,k4)
        if RouteCost == best[1] and not i == best[0]:
            extras.append([i, RouteCost])
        
        if RouteCost < best[1]:
            extras = []
            best = [i, RouteCost]
    return best, extras

In [25]:
l = 4
w = 4
D = 5
adj_matrix = GenGrid(l,w)
Demand = FixedDemand(l,w,D)

k1 = 100    #Bus operating cost
k2 = 25     #Walk cost
k3 = 5      #Bus travel cost
k4 = 5      #Bus ticket cost

Destination = 15
ExhaustiveOptimal(adj_matrix, Demand, k1,k2,k3,k4, Destination)

([[2, 1, 5, 9, 13, 14, 15], 3325], [[[8, 4, 5, 6, 7, 11, 15], 3325]])